# Gifi Theory

Python equivalent of R Gifi vignette: *Gifi Theory*

---

## Overview

The **Gifi system** (De Leeuw & Mair, 2009; Gifi, 1990) is a unified framework for
multivariate analysis with **optimal scaling** — a technique that transforms
categorical, ordinal, or mixed-type variables into numeric representations that
maximise some criterion (usually variance explained or minimised stress).

The key models in the system:

| Model | Full Name | Purpose |
|-------|-----------|--------|
| **Homals** | Homogeneity Analysis | Multiple Correspondence Analysis (MCA) |
| **Princals** | Principal Components with Optimal Scaling | Ordinal / mixed PCA |
| **Morals** | Multiple Optimal Regression with Alternating Least Squares | Optimal scaling regression |
| **Corals** | Canonical Correlation with Optimal Scaling | Canonical analysis |
| **Overals** | Multiset Canonical Correlation | Generalised multiset analysis |

## 1 · The Gifi Loss Function

All Gifi models minimise a **stress** criterion of the form:

$$
\sigma(\mathbf{X}, \{\mathbf{Z}_j\}, \{\mathbf{A}_j\}) 
= \sum_{j=1}^{m} \| \mathbf{X} - \mathbf{G}_j \mathbf{Z}_j \mathbf{A}_j \|^2
$$

where:
- $\mathbf{X}$ is the $n \times p$ matrix of **object scores** (low-dimensional representation).
- $\mathbf{G}_j$ is the $n \times k_j$ **indicator matrix** for variable $j$ (one column per category).
- $\mathbf{Z}_j$ is the $k_j \times s_j$ matrix of **category quantifications**.
- $\mathbf{A}_j$ is the $s_j \times p$ matrix of **component loadings** (weights).

The product $\mathbf{Y}_j = \mathbf{G}_j \mathbf{Z}_j$ gives the **optimal transformation** of variable $j$,
and $\mathbf{Y}_j \mathbf{A}_j$ projects it into the low-dimensional space.

Minimising $\sigma$ over $\mathbf{X}$, $\mathbf{Z}_j$, and $\mathbf{A}_j$ subject to constraints
defines the specific Gifi model.

## 2 · Measurement Levels and Cone Constraints

The feasible set for each $\mathbf{Z}_j$ is a **cone** determined by the measurement level:

| Level | Cone Type | Constraint on $\mathbf{Z}_j$ |
|-------|-----------|-------------------------------|
| **Nominal** | Subspace | $\mathbf{Z}_j$ unrestricted (free OLS onto category indicators) |
| **Ordinal** | Isotone cone | $\mathbf{Z}_j$ must be non-decreasing (PAVA enforced) |
| **Numerical** | Linear | $\mathbf{Z}_j$ constrained to linear function of original values |
| **Spline** | Spline subspace ∩ isotone | B-spline basis + monotone coefficients (Dykstra's algorithm) |

The cone projection step is performed by PyGiFi's `project_cone()` function (in `pygifi/utils/_cone.py`).

## 3 · The ALS Algorithm

Gifi models are fitted by **Alternating Least Squares (ALS)**, cycling through three update steps:

```
Initialise X (random orthonormal)
repeat until convergence:
    1. Z-step: for each variable j, project G_j^T X onto its cone → Z_j
    2. A-step: for each variable j, A_j ← (Z_j^T G_j^T G_j Z_j)^{-1} Z_j^T G_j^T X
    3. X-step: X ← Gram-Schmidt-orthonormalised sum of G_j Z_j A_j
    4. Compute loss σ; stop if Δσ < ε
```

ALS is **guaranteed to decrease loss** at every step but may converge to a local (not global) minimum for non-convex cone types (ordinal, spline).

> **RNG note**: Because ALS depends on the random initialisation of X, reproducible results require seeding via the `init_x` parameter.

## 4 · Python Demo: the stress surface

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pygifi import Homals, get_dataset

# Load ABC satisfaction data (nominal, n=2176, 3 variables)
abc = get_dataset('abc')
print(abc.head())
print(f'Shape: {abc.shape}')

In [ ]:
# Fit Homals and inspect the loss surface via iteration history
model = Homals(ndim=2, eps=1e-8, itmax=500).fit(abc)
print(model)

In [ ]:
# Show the optimal transformations (category quantifications)
for var, quant in model.result_['quantifications'].items():
    print(f'\nVariable: {var}')
    print(np.round(quant, 4))

## 5 · Eigenvalue Structure

After convergence, the eigenvalues $\lambda_k$ measure how much variance is explained by each dimension.

In [ ]:
evals = model.eigenvalues_
var_exp = evals / evals.sum() * 100

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(1, len(evals)+1), evals, color='steelblue', alpha=0.8)
ax.set_xlabel('Dimension')
ax.set_ylabel('Eigenvalue')
ax.set_title('Homals Scree Plot — ABC Data')
for i, (e, v) in enumerate(zip(evals, var_exp)):
    ax.text(i+1, e+0.005, f'{v:.1f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()
print('\nEigenvalues:', np.round(evals, 4))
print('% Variance:', np.round(var_exp, 2))

## 6 · PAVA and Isotone Regression

For ordinal variables, Gifi applies the **Pool Adjacent Violators Algorithm (PAVA)** to enforce monotonicity on category quantifications.
PyGiFi implements this as `pygifi.utils.isotone.pava()`.

In [ ]:
from pygifi.utils.isotone import pava, isotone
import numpy as np

# Example: enforce isotone regression on arbitrary values
y_raw = np.array([3.0, 1.5, 2.0, 4.5, 3.5, 5.0])
y_iso = pava(y_raw)

print('Raw values:    ', y_raw)
print('Isotone values:', np.round(y_iso, 4))

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(y_raw, 'o--', label='raw', alpha=0.7)
ax.step(range(len(y_iso)), y_iso, where='mid', label='isotone (PAVA)', lw=2)
ax.legend()
ax.set_title('Pool Adjacent Violators (PAVA)')
plt.tight_layout()
plt.show()

## 7 · References

- Gifi, A. (1990). *Nonlinear Multivariate Analysis*. Wiley.
- De Leeuw, J., & Mair, P. (2009). Gifi methods for optimal scaling in R: The package homals. *Journal of Statistical Software*, 31(4).
- Mair, P., De Leeuw, J., & Groenen, P. J. F. (2022). *Gifi: Multivariate Analysis with Optimal Scaling*. CRAN package.
- Kruskal, J. B. (1964). Multidimensional scaling by optimizing goodness of fit to a nonmetric hypothesis. *Psychometrika*, 29(1), 1–27.